<a href="https://colab.research.google.com/github/narendrapatel6321-dotcom/sec-10k-crag/blob/main/notebooks/sec_10k_rag_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SEC 10-K RAG — Automated Evaluation
Clones the deployed repo, installs dependencies, and runs the CRAG benchmark
suite against the live Pinecone index + Groq pipeline.

## Clone Repo & Install Dependencies

In [1]:
import os
from google.colab import userdata

REPO_URL = "https://github.com/narendrapatel6321-dotcom/sec-10k-crag.git"
REPO_DIR = "/content/financial-sec-rag"

try:
    !git clone {REPO_URL} {REPO_DIR}
except:
    print("Repo already cloned, pulling latest...")
    !cd {REPO_DIR} && git pull

Cloning into '/content/financial-sec-rag'...
remote: Enumerating objects: 161, done.
remote: Counting objects: 100% (161/161), done.
remote: Compressing objects: 100% (135/135), done.
remote: Total 161 (delta 51), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (161/161), 2.36 MiB | 4.18 MiB/s, done.
Resolving deltas: 100% (51/51), done.


In [2]:
%cd {REPO_DIR}


!pip install -q --upgrade \
    "numpy<2.1" \
    "pandas==2.2.2" \
    "requests==2.32.4" \
    edgartools \
    sec-edgar-downloader \
    langchain \
    langchain-core \
    langchain-community \
    langchain-pinecone \
    pinecone \
    langchain-huggingface \
    sentence-transformers \
    rank_bm25 \
    langgraph \
    langgraph-prebuilt \
    langgraph-sdk

/content/financial-sec-rag
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.3/611.3 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.4/155.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

## Set API Keys

In [3]:
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["PINECONE_API_KEY"] = userdata.get("PINECONE_API_KEY")
os.environ["PINECONE_INDEX_NAME"] = userdata.get("PINECONE_INDEX_NAME")

print("Keys loaded:", all([
    os.environ.get("GROQ_API_KEY"),
    os.environ.get("PINECONE_API_KEY"),
    os.environ.get("PINECONE_INDEX_NAME"),
]))

Keys loaded: True


## Verify Import Path & BM25 Index Presence

In [4]:
import sys
sys.path.insert(0, REPO_DIR)

from pathlib import Path
bm25_path = Path(REPO_DIR) / "data" / "index" / "bm25_retriever.pkl"
assert bm25_path.exists(), f"BM25 index missing at {bm25_path} — pull it from Drive and commit it first."
print(f"BM25 index found: {bm25_path} ({bm25_path.stat().st_size / 1024:.1f} KB)")

BM25 index found: /content/financial-sec-rag/data/index/bm25_retriever.pkl (9652.8 KB)


## Run the Benchmark

In [8]:
!pip -q install langchain_groq langchain_classic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 13.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.30 requires langchain-core<1.0.0,>=0.3.85, but you have langchain-core 1.5.5 which is incompatible.
langchain 0.3.30 requires langchain-text-splitters<1.0.0,>=0.3.9, but you have langchain-text-splitters 1.1.2 which is incompatible.


In [9]:
from src.eval.benchmark import run_benchmark

results_df = run_benchmark(output_path="benchmark_results.csv")
results_df

Starting Benchmark Evaluation (7 test cases)...

[TC-01] Evaluating: 'What are the primary operational and regulatory risk factors...'
---NODE: ROUTE QUERY---
  -> category=sec_filing ticker=C section=risk_factors
---ROUTING QUERY---
  -> Routing to SEC 10-K Retrieval (Ticker: C)
---NODE: RETRIEVE---


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

---NODE: GRADE DOCUMENTS---
---CHECKING RELEVANCE---
  -> Documents are relevant. Proceeding to generation.
---NODE: GENERATE---
---NODE: VERIFY NUMBERS---
---CHECKING VERIFICATION---
  -> Verification passed. Ending workflow.
  -> Done in 68.21s | Route: PASS | Faithfulness: 1.0 | Numeric: 1.0
[TC-03] Evaluating: 'What is Microsoft's MD&A commentary regarding cloud segment ...'
---NODE: ROUTE QUERY---
  -> category=sec_filing ticker=MSFT section=mdna
---ROUTING QUERY---
  -> Routing to SEC 10-K Retrieval (Ticker: MSFT)
---NODE: RETRIEVE---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

---NODE: GRADE DOCUMENTS---
---CHECKING RELEVANCE---
  -> Documents are relevant. Proceeding to generation.
---NODE: GENERATE---
---NODE: VERIFY NUMBERS---
---CHECKING VERIFICATION---
  -> Verification passed. Ending workflow.
  -> Done in 72.68s | Route: PASS | Faithfulness: 0.9 | Numeric: 1.0
[TC-04] Evaluating: 'How is the Common Equity Tier 1 (CET1) ratio calculated unde...'
---NODE: ROUTE QUERY---
  -> category=general_financial ticker=None section=None
---ROUTING QUERY---
  -> Routing to standard generation (Category: general_financial)
  -> Done in 1.43s | Route: PASS | Faithfulness: 1.0 | Numeric: 1.0
[TC-05] Evaluating: 'Can you write Python code to train a Convolutional Neural Ne...'
---NODE: ROUTE QUERY---
  -> category=out_of_scope ticker=None section=None
---ROUTING QUERY---
  -> Routing to standard generation (Category: out_of_scope)
  -> Done in 0.94s | Route: PASS | Faithfulness: 1.0 | Numeric: 1.0
[TC-06] Evaluating: 'What was Apple's total net sales for the most recen

,id,type,question,route_passed,predicted_ticker,predicted_category,retrieved_chunks,numeric_accuracy,faithfulness_score,is_faithful,judge_reasoning,latency_seconds,rewrite_retries,verify_retries,generation
0,TC-01,qualitative_risk,What are the primary operational and regulator...,True,C,sec_filing,5,1.0,1.0,True,The generated answer is strictly faithful to a...,92.76,0,1,Based on the provided context (C | mdna | acce...
1,TC-02,qualitative_risk,Discuss Apple's supply chain concentration and...,True,AAPL,sec_filing,5,1.0,1.0,True,The generated answer is strictly faithful to a...,68.21,0,1,"According to the provided context (AAPL, risk_..."
2,TC-03,qualitative_mdna,What is Microsoft's MD&A commentary regarding ...,True,MSFT,sec_filing,3,1.0,0.9,True,The generated answer is largely faithful to th...,72.68,0,1,"According to the provided context (MSFT, mdna,..."
3,TC-04,general_finance,How is the Common Equity Tier 1 (CET1) ratio c...,True,None,general_financial,0,1.0,1.0,True,No context required for query route.,1.43,0,0,The Common Equity Tier 1 (CET1) ratio is a cru...
4,TC-05,out_of_scope,Can you write Python code to train a Convoluti...,True,None,out_of_scope,0,1.0,1.0,True,No context required for query route.,0.94,0,0,I'm happy to help with any questions related t...
5,TC-06,quantitative_mdna,What was Apple's total net sales for the most ...,True,AAPL,sec_filing,3,1.0,1.0,True,The answer directly quotes the total net sales...,90.20,0,1,According to the provided context (AAPL | mdna...
6,TC-07,quantitative_mdna,What was Citigroup's Common Equity Tier 1 (CET...,True,C,sec_filing,4,1.0,1.0,True,The answer directly quotes the CET1 Capital ra...,86.57,0,1,According to the provided context (C | mdna | ...


## Inspect Failures

In [10]:
failures = results_df[
    (~results_df["route_passed"]) |
    (results_df["faithfulness_score"] < 0.7) |
    (results_df["numeric_accuracy"] < 1.0)
]
failures[["id", "type", "route_passed", "faithfulness_score", "numeric_accuracy", "judge_reasoning"]]

,id,type,route_passed,faithfulness_score,numeric_accuracy,judge_reasoning


## Save Results to Drive

In [11]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/sec-10k-rag/eval_results"
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

import datetime
timestamped_name = f"benchmark_results_{datetime.datetime.now():%Y%m%d_%H%M%S}.csv"
shutil.copy("benchmark_results.csv", os.path.join(DRIVE_RESULTS_DIR, timestamped_name))
print(f"Saved to {DRIVE_RESULTS_DIR}/{timestamped_name}")

Mounted at /content/drive
Saved to /content/drive/MyDrive/sec-10k-rag/eval_results/benchmark_results_20260815_130057.csv
